# Tahap 6: Rekomendasi strategi per segmen

Notebook ini hanya memanggil `umkm.rekomendasi.run()` lalu menampilkan hasilnya. Logika ada di
`src/umkm/recommend.py` (inferensi), `rekomendasi.py` (analisis), dan `rekomendasi_teks.py`.
Jalankan ulang lewat CLI: `python -m umkm.pipeline --stage 6` (sekitar 10 detik).

Rumus (D28): gap = max(0, rata-rata skor UMKM yang naik di segmen - rata-rata skor segmen);
prioritas = bobot Tahap 4 x gap. Hasilnya prioritas berbasis asosiasi, bukan jaminan hasil.

In [ ]:
import warnings

from IPython.display import Image, Markdown, display

from umkm import config, rekomendasi
from umkm.rekomendasi_teks import interpretasi

warnings.filterwarnings("ignore")
rep = rekomendasi.run()
reports = config.path("reports")
print(f"model {rep.info['model_terbaik']}, {len(rep.tabel['per_segmen'])} baris rekomendasi, "
      f"recommend() cocok dengan tabel: {bool(rep.tabel['konsistensi']['cocok'].all())}")

## Ringkasan

In [ ]:
display(Markdown((reports / "rekomendasi_ringkas.md").read_text(encoding="utf-8")))

## Grafik dan interpretasi

In [ ]:
interp = interpretasi(rep)
for nama, path in rep.grafik.items():
    display(Markdown(f"### {nama}"))
    display(Image(filename=str(path)))
    display(Markdown(f"**Interpretasi:** {interp[nama]}"))

## Tabel kunci

In [ ]:
rep.tabel["per_segmen"][["segmen", "peringkat", "nama_strategi", "skor_segmen", "benchmark", "gap",
                         "bobot", "prioritas", "p_tiga_besar", "fokus_butir"]].round(3)

In [ ]:
rep.tabel["sensitivitas"][["keterangan", "segmen", "urutan", "sama_dengan_utama", "rho_vs_utama"]]

In [ ]:
rep.tabel["konsistensi"]

## Mencoba `recommend()` untuk satu UMKM

Fungsi ini hanya memuat artefak `models/v1/` (tanpa data survei). Kontrak lengkap:
`reports/integration_contract.md`.

In [ ]:
import json

from umkm.recommend import recommend

hasil = recommend({
    "omzet_bulanan": 12_000_000,
    "lama_usaha_pra_mp": 2,
    "lama_di_mp": 1,
    "platform": ["gofood", "shopee"],
    "x1_unggah": 2.25,
    "x2_promosi": 2.0,
    "x3_komunikasi": 3.2,
    "x4_konsistensi": 3.0,
})
print(json.dumps(hasil, ensure_ascii=False, indent=2))